<a href="https://colab.research.google.com/github/SreeGayatri369/meetily/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q crewai yfinance feedparser ddgs pandas numpy pyyaml transformers

In [2]:
import os, sys
from pathlib import Path

ROOT = Path("/content/stock_platform")
for d in ["config", "src/stockcrew", "runs", "cache"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

from google.colab import userdata
for k in ["GROQ_API_KEY", "HF_TOKEN"]:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print(f"[info] {k} not set")

os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
sys.path.insert(0, str(ROOT / "src"))

[info] HF_TOKEN not set


In [3]:

%%writefile /content/stock_platform/config/settings.yaml
llm:
  provider_order: [groq, ollama, huggingface]   # tried in order, auto-fallback on failure
  groq_worker_model: groq/llama-3.3-70b-versatile   # switch to a smaller model if you hit rate limits
  groq_manager_model: groq/llama-3.3-70b-versatile
  ollama_model: ollama/llama3.1:8b
  ollama_base_url: http://localhost:11434
  hf_model: huggingface/Qwen/Qwen2.5-72B-Instruct   # availability on free HF inference changes; verify
  temperature: 0.1
  max_tokens: 2048
crew:
  max_rpm: 15
  max_iter: 8
news:
  region: US
  language: en
  lookback: 3d
  max_items_per_query: 25
  max_headlines_returned: 40
  company_news_items: 15
  cache_ttl_minutes: 30
discovery:
  max_candidates: 8
  name_match_min: 0.6
market:
  history_period: 1y
  interval: 1d
  trading_days: 252
  trend_sma_days: 200
  windows: {ret_1m: 21, ret_3m: 63, ret_6m: 126}
  cache_ttl_minutes: 60
validation:
  price_tolerance: 0.05
sentiment:
  backend: finbert          # finbert | none
  model: ProsusAI/finbert
scoring:
  min_peer_group_size: 3    # rank within sector only if enough peers, else across all candidates
  risk_flag_percentile: 0.2

Overwriting /content/stock_platform/config/settings.yaml


In [4]:
%%writefile /content/stock_platform/config/factors.yaml
# Edit this file to change the scoring model. No code changes needed.
# source: info (yfinance info) | tech (price history) | derived | news | trend
risk_families: [risk]
families:
  valuation:
    weight: 1
    metrics:
      trailingPE:       {direction: lower,  source: info,    positive_only: true}
      forwardPE:        {direction: lower,  source: info,    positive_only: true}
      priceToBook:      {direction: lower,  source: info,    positive_only: true}
      upside_to_target: {direction: higher, source: derived}
  quality:
    weight: 1
    metrics:
      profitMargins:    {direction: higher, source: info}
      operatingMargins: {direction: higher, source: info}
      returnOnEquity:   {direction: higher, source: info}
      currentRatio:     {direction: higher, source: info}
  growth:
    weight: 1
    metrics:
      revenueGrowth:    {direction: higher, source: info}
      earningsGrowth:   {direction: higher, source: info}
  momentum:
    weight: 1
    metrics:
      ret_1m:     {direction: higher, source: tech}
      ret_3m:     {direction: higher, source: tech}
      ret_6m:     {direction: higher, source: tech}
      above_sma:  {direction: higher, source: tech}
  risk:
    weight: 1
    metrics:
      volatility:    {direction: lower, source: tech}
      max_drawdown:  {direction: lower, source: tech}
      beta:          {direction: lower, source: info}
      debtToEquity:  {direction: lower, source: info, positive_only: true}
  sentiment:
    weight: 1
    metrics:
      news_sentiment: {direction: higher, source: news}
  trend:
    weight: 1
    metrics:
      news_mentions:   {direction: higher, source: trend}
      news_publishers: {direction: higher, source: trend}
  analyst:
    weight: 1
    metrics:
      recommendationMean: {direction: lower, source: info}

Overwriting /content/stock_platform/config/factors.yaml


In [5]:
%%writefile /content/stock_platform/config/agents.yaml
manager:
  role: "Chief Investment Officer"
  goal: "Coordinate the research team to deliver ONE evidence-backed stock recommendation for a {risk_profile} investor with a {horizon} horizon. Research focus: {focus}."
  backstory: "You run a disciplined research desk. You delegate each step to the right specialist, check that every step produced real tool output, and never fabricate data. You insist on the order: discover, resolve, fundamentals, risk, validate, rank, report."

news_scout:
  role: "Live Market News Scout"
  goal: "Find which publicly traded companies are being discussed in current financial news for this focus: {focus} (region: {region})."
  backstory: "You monitor live news. You only report companies that literally appear in headlines returned by your tool. You never invent or assume companies."
  tools: [search_live_news]

ticker_resolver:
  role: "Entity and Ticker Resolution Specialist"
  goal: "Convert company names found in headlines into verified exchange-listed tickers."
  backstory: "You are careful about name ambiguity and private or unlisted companies. You rely only on your resolver tool and report what could not be resolved."
  tools: [resolve_tickers]

fundamental_analyst:
  role: "Fundamental Financial Analyst"
  goal: "Collect valuation, quality, growth and analyst-consensus metrics for all candidate companies."
  backstory: "You rely strictly on tool-provided data and clearly state when data is missing."
  tools: [analyze_fundamentals]

risk_analyst:
  role: "Market and Risk Analyst"
  goal: "Measure price momentum, volatility, drawdown and news sentiment for all candidate companies."
  backstory: "You think in terms of downside first. You only report numbers returned by your tool."
  tools: [analyze_market_risk]

validation_analyst:
  role: "Data Validation and Fact-Check Analyst"
  goal: "Verify data integrity and corroborate news coverage across multiple independent sources for every candidate."
  backstory: "You are skeptical. You report inconsistencies and weak corroboration explicitly instead of smoothing them over."
  tools: [validate_candidates]

portfolio_strategist:
  role: "Portfolio Strategist"
  goal: "Rank candidates objectively using the quantitative ranking engine for a {risk_profile} investor."
  backstory: "You never override the ranking engine's numbers. You interpret them."
  tools: [rank_candidates]

research_writer:
  role: "Investment Research Writer"
  goal: "Write a clear, evidence-based recommendation that cites concrete numbers and headlines from tool output."
  backstory: "You only use facts returned by the evidence tool. You always include risks and data limitations."
  tools: [get_evidence]

Overwriting /content/stock_platform/config/agents.yaml


In [6]:
%%writefile /content/stock_platform/config/tasks.yaml
discover:
  description: >
    Delegate to the Live Market News Scout. Use search_live_news with at least three
    DIFFERENT queries derived from the focus "{focus}" and region "{region}" (for example
    earnings, guidance, regulation, product news, analyst actions, market movers; if the focus
    is broad, query generally). Then list the distinct publicly traded companies that
    literally appear in the returned headlines.
  expected_output: "A JSON list of company names exactly as they appear in headlines."

resolve:
  description: >
    Delegate to the Entity and Ticker Resolution Specialist. Call resolve_tickers ONCE with all
    company names from the previous step. Report resolved tickers and unresolved names.
  expected_output: "List of resolved tickers with company names, plus unresolved names."
  context: [discover]

fundamentals:
  description: >
    Delegate to the Fundamental Financial Analyst. Call analyze_fundamentals (omit tickers to
    use all resolved candidates). Summarize data coverage.
  expected_output: "Per-ticker data coverage summary for fundamentals."
  context: [resolve]

risk:
  description: >
    Delegate to the Market and Risk Analyst. Call analyze_market_risk (omit tickers to use all
    candidates). Summarize coverage.
  expected_output: "Per-ticker coverage summary for market, risk and sentiment metrics."
  context: [resolve]

validate:
  description: >
    Delegate to the Data Validation Analyst. Call validate_candidates and report validation
    scores and any issues found.
  expected_output: "Per-ticker validation score and list of issues."
  context: [fundamentals, risk]

rank:
  description: >
    Delegate to the Portfolio Strategist. Call rank_candidates and return the ranked table.
  expected_output: "Ranked table with final score, coverage, validation score, and family scores."
  context: [validate]

report:
  description: >
    Delegate to the Investment Research Writer. Call get_evidence for the top-ranked companies,
    then produce the final recommendation for a {risk_profile} investor, {horizon} horizon.
    The recommended ticker MUST be the rank-1 company unless the evidence shows a concrete
    data-integrity problem, in which case say so explicitly. Cite specific numbers and headlines
    from the evidence. Include key risks and data limitations. Use only facts from tool output.
  expected_output: >
    A structured recommendation: recommended_ticker, company_name, confidence (0-1), thesis,
    supporting_evidence, key_risks, runner_ups.
  context: [rank]

Overwriting /content/stock_platform/config/tasks.yaml


In [7]:
%%writefile /content/stock_platform/src/stockcrew/__init__.py

Overwriting /content/stock_platform/src/stockcrew/__init__.py


In [8]:
%%writefile /content/stock_platform/src/stockcrew/config.py
import os
from pathlib import Path
import yaml

ROOT = Path(os.environ.get("STOCKCREW_HOME", "/content/stock_platform"))
CONFIG_DIR, CACHE_DIR, RUNS_DIR = ROOT / "config", ROOT / "cache", ROOT / "runs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def load_yaml(name):
    with open(CONFIG_DIR / name) as f:
        return yaml.safe_load(f)

SETTINGS = load_yaml("settings.yaml")
FACTORS = load_yaml("factors.yaml")

Overwriting /content/stock_platform/src/stockcrew/config.py


In [9]:
%%writefile /content/stock_platform/src/stockcrew/utils.py
import functools, hashlib, json, math, re, time
from .config import CACHE_DIR

def retry(times=3, base=1.5):
    def deco(fn):
        @functools.wraps(fn)
        def wrap(*a, **k):
            for i in range(times):
                try:
                    return fn(*a, **k)
                except Exception:
                    if i == times - 1:
                        raise
                    time.sleep(base * (2 ** i))
        return wrap
    return deco

def _cpath(key):
    return CACHE_DIR / (hashlib.md5(key.encode()).hexdigest() + ".json")

def cache_get(key, ttl_min):
    p = _cpath(key)
    if p.exists() and (time.time() - p.stat().st_mtime) / 60 <= ttl_min:
        try:
            return json.loads(p.read_text())
        except Exception:
            return None
    return None

def cache_set(key, val):
    try:
        _cpath(key).write_text(json.dumps(val, default=str))
    except Exception:
        pass

def clean_num(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) else None

_SUFFIX = re.compile(
    r"\b(inc|incorporated|corp|corporation|co|company|ltd|limited|plc|llc|lp|holdings?|"
    r"group|sa|ag|nv|se|the|class [a-c]|common stock|ordinary shares?)\b")

def normalize_name(s):
    s = re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())
    s = _SUFFIX.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def as_list(x):
    """Tolerant parser: LLMs pass lists, JSON strings or comma-separated text."""
    if x is None:
        return []
    def item(i):
        if isinstance(i, dict):
            i = i.get("name") or i.get("ticker") or next(iter(i.values()), "")
        return str(i).strip()
    if isinstance(x, (list, tuple)):
        return [item(i) for i in x if item(i)]
    s = str(x).strip()
    try:
        v = json.loads(s)
        if isinstance(v, list):
            return [item(i) for i in v if item(i)]
    except Exception:
        pass
    return [p.strip().strip("\"'") for p in re.split(r"[,\n;]", s.strip("[]")) if p.strip()]

Overwriting /content/stock_platform/src/stockcrew/utils.py


In [10]:
%%writefile /content/stock_platform/src/stockcrew/store.py
import json, time
from .config import RUNS_DIR

class RunStore:
    """Shared, file-backed state between tools/agents. Resumable and auditable."""
    def __init__(self, run_id=None):
        self.run_id = run_id or time.strftime("%Y%m%d-%H%M%S")
        self.dir = RUNS_DIR / self.run_id
        self.dir.mkdir(parents=True, exist_ok=True)

    def save(self, key, obj):
        (self.dir / f"{key}.json").write_text(json.dumps(obj, indent=2, default=str))

    def load(self, key, default=None):
        p = self.dir / f"{key}.json"
        return json.loads(p.read_text()) if p.exists() else default

_STORE = None

def init_run(run_id=None):
    global _STORE
    _STORE = RunStore(run_id)
    return _STORE

def store():
    return _STORE or init_run()

Overwriting /content/stock_platform/src/stockcrew/store.py


In [11]:
%%writefile /content/stock_platform/src/stockcrew/schemas.py
from typing import List
from pydantic import BaseModel, Field

class FinalRecommendation(BaseModel):
    recommended_ticker: str
    company_name: str
    confidence: float = Field(ge=0, le=1, description="0-1 confidence in the recommendation")
    thesis: str
    supporting_evidence: List[str]
    key_risks: List[str]
    runner_ups: List[str] = []

Overwriting /content/stock_platform/src/stockcrew/schemas.py


In [12]:
%%writefile /content/stock_platform/src/stockcrew/data_sources.py
import difflib, urllib.parse
import feedparser
import numpy as np
import pandas as pd
import yfinance as yf
from .config import SETTINGS, FACTORS
from .utils import retry, cache_get, cache_set, normalize_name

EXTRA_INFO = ["longName", "shortName", "sector", "industry", "currency",
              "currentPrice", "targetMeanPrice", "marketCap"]

def info_keys():
    keys = [m for f in FACTORS["families"].values()
            for m, s in f["metrics"].items() if s.get("source") == "info"]
    return keys + EXTRA_INFO

# ---------- News ----------
@retry(times=3)
def fetch_news(query, max_items=None):
    n = SETTINGS["news"]
    max_items = max_items or n["max_items_per_query"]
    key = f"news::{query}::{n['lookback']}::{n['region']}::{max_items}"
    hit = cache_get(key, n["cache_ttl_minutes"])
    if hit is not None:
        return hit
    q = f"{query} when:{n['lookback']}" if n.get("lookback") else query
    url = ("https://news.google.com/rss/search?q=" + urllib.parse.quote(q) +
           f"&hl={n['language']}-{n['region']}&gl={n['region']}&ceid={n['region']}:{n['language']}")
    feed = feedparser.parse(url)
    items, seen = [], set()
    for e in feed.entries:
        t = (e.get("title") or "").strip()
        if not t or t in seen:
            continue
        seen.add(t)
        src = (e.get("source") or {}).get("title", "") if e.get("source") else ""
        items.append({"title": t, "publisher": src, "link": e.get("link"),
                      "published": e.get("published", "")})
        if len(items) >= max_items:
            break
    cache_set(key, items)
    return items

def ddg_news(query, n=10):
    """Independent second news source (best effort)."""
    try:
        try:
            from ddgs import DDGS
        except ImportError:
            from duckduckgo_search import DDGS
        res = DDGS().news(query, max_results=n) or []
        return [{"title": r.get("title", ""), "publisher": r.get("source", ""),
                 "link": r.get("url", "")} for r in res]
    except Exception:
        return []

# ---------- Sentiment (free HF model, lazy-loaded) ----------
_PIPE = None

def score_headlines(titles):
    if SETTINGS["sentiment"]["backend"] != "finbert" or not titles:
        return None
    global _PIPE
    try:
        if _PIPE is None:
            import torch
            from transformers import pipeline
            _PIPE = pipeline("text-classification", model=SETTINGS["sentiment"]["model"],
                             top_k=None, truncation=True,
                             device=0 if torch.cuda.is_available() else -1)
        vals = []
        for o in _PIPE(list(titles)):
            d = {x["label"].lower(): x["score"] for x in o}
            vals.append(d.get("positive", 0.0) - d.get("negative", 0.0))
        return float(np.mean(vals))
    except Exception:
        return None

# ---------- Entity resolution ----------
@retry(times=3)
def _yf_search(name):
    return yf.Search(name, max_results=8).quotes

def resolve_company(name):
    target = normalize_name(name)
    best = None
    for q in _yf_search(name) or []:
        if q.get("quoteType") != "EQUITY":
            continue
        for cand in (q.get("longname"), q.get("shortname")):
            if cand:
                r = difflib.SequenceMatcher(None, target, normalize_name(cand)).ratio()
                if best is None or r > best[0]:
                    best = (r, q)
    if best and best[0] >= SETTINGS["discovery"]["name_match_min"]:
        q = best[1]
        return {"input": name, "ticker": q["symbol"],
                "company": q.get("longname") or q.get("shortname"),
                "exchange": q.get("exchange"), "match": round(best[0], 3)}
    return None

def count_mentions(variants, headlines):
    toks = [normalize_name(v) for v in variants if v]
    hits = [h for h in headlines
            if any(t and f" {t} " in f" {normalize_name(h['title'])} " for t in toks)]
    pubs = {h.get("publisher") for h in hits if h.get("publisher")}
    return len(hits), len(pubs)

# ---------- Market data ----------
def tech_metrics(closes):
    m = SETTINGS["market"]
    out = {}
    if len(closes) < 2:
        return out
    for name, days in m["windows"].items():
        if len(closes) > days:
            out[name] = float(closes.iloc[-1] / closes.iloc[-1 - days] - 1)
    r = closes.pct_change().dropna()
    out["volatility"] = float(r.std() * np.sqrt(m["trading_days"]))
    out["max_drawdown"] = float(abs((closes / closes.cummax() - 1).min()))
    d = m["trend_sma_days"]
    if len(closes) >= d:
        out["above_sma"] = float(closes.iloc[-1] / closes.iloc[-d:].mean() - 1)
    return out

@retry(times=3, base=2.0)
def get_snapshot(ticker):
    m = SETTINGS["market"]
    key = f"snap::{ticker}::{m['history_period']}"
    hit = cache_get(key, m["cache_ttl_minutes"])
    if hit is not None:
        return hit
    t = yf.Ticker(ticker)
    info = t.info or {}
    hist = t.history(period=m["history_period"], interval=m["interval"], auto_adjust=True)
    closes = hist["Close"].dropna() if not hist.empty else pd.Series(dtype=float)
    snap = {"info": {k: info.get(k) for k in info_keys()},
            "last_close": float(closes.iloc[-1]) if len(closes) else None,
            "tech": tech_metrics(closes)}
    cache_set(key, snap)
    return snap

Overwriting /content/stock_platform/src/stockcrew/data_sources.py


In [13]:
%%writefile /content/stock_platform/src/stockcrew/scoring.py
import numpy as np
import pandas as pd
from .config import FACTORS, SETTINGS

def metric_specs():
    return [(fam, m, spec) for fam, fd in FACTORS["families"].items()
            for m, spec in fd["metrics"].items()]

def percentile(s, direction):
    v = s.dropna()
    if len(v) == 0:
        return s * np.nan
    out = pd.Series(0.5, index=v.index) if len(v) == 1 else (v.rank(method="average") - 1) / (len(v) - 1)
    if direction == "lower":
        out = 1 - out
    return out.reindex(s.index)

def grouped_percentile(s, direction, groups, min_size):
    glob = percentile(s, direction)
    if groups is None:
        return glob
    out = pd.Series(np.nan, index=s.index)
    for _, idx in groups.dropna().groupby(groups.dropna()).groups.items():
        sub = s.loc[idx].dropna()
        if len(sub) >= min_size:
            out.loc[sub.index] = percentile(sub, direction)
    return out.fillna(glob)

def score(df, groups=None, validation=None, weight_overrides=None):
    cfg = SETTINGS["scoring"]
    weights = {f: float(d.get("weight", 1)) for f, d in FACTORS["families"].items()}
    for f, w in (weight_overrides or {}).items():
        if f in weights:
            weights[f] = float(w)
    specs = metric_specs()
    pct = {}
    for fam, m, spec in specs:
        if m not in df.columns:
            continue
        s = pd.to_numeric(df[m], errors="coerce")
        if spec.get("positive_only"):
            s = s.where(s > 0)
        pct[(fam, m)] = grouped_percentile(s, spec["direction"], groups, cfg["min_peer_group_size"])
    risk_fams = set(FACTORS.get("risk_families", []))
    out = []
    for t in df.index:
        per_fam, per_metric = {}, []
        for (fam, m), s in pct.items():
            v = s.get(t)
            if v is not None and pd.notna(v):
                per_fam.setdefault(fam, []).append(float(v))
                per_metric.append((fam, m, float(v)))
        fs = {f: float(np.mean(v)) for f, v in per_fam.items()}
        wsum = sum(weights[f] for f in fs)
        comp = sum(weights[f] * x for f, x in fs.items()) / wsum if wsum > 0 else 0.0
        cov = len(per_metric) / len(specs)
        val = (validation or {}).get(t, 1.0)
        per_metric.sort(key=lambda x: x[2])
        flags = [f"{m} = {df.loc[t, m]:.3g} is riskier than {1 - v:.0%} of peers"
                 for fam, m, v in per_metric
                 if fam in risk_fams and v <= cfg["risk_flag_percentile"]]
        out.append({
            "ticker": t, "composite": round(comp, 4), "coverage": round(cov, 3),
            "validation": round(val, 3), "final": round(comp * cov * val, 4),
            "families": {f: round(x, 3) for f, x in fs.items()},
            "strengths": [f"{m} (top {1 - v:.0%} tier, raw={df.loc[t, m]:.3g})" for _, m, v in per_metric[-3:][::-1]],
            "weaknesses": [f"{m} (percentile {v:.0%}, raw={df.loc[t, m]:.3g})" for _, m, v in per_metric[:3]],
            "risk_flags": flags,
            "raw": {m: (None if pd.isna(df.loc[t, m]) else float(df.loc[t, m]))
                    for m in df.columns if pd.api.types.is_number(df.loc[t, m])},
        })
    out.sort(key=lambda r: r["final"], reverse=True)
    for i, r in enumerate(out, 1):
        r["rank"] = i
    return out

Overwriting /content/stock_platform/src/stockcrew/scoring.py


In [14]:
%%writefile /content/stock_platform/src/stockcrew/tools.py
import json
from typing import Optional, Type, Union, List
import pandas as pd
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

from .config import SETTINGS, FACTORS
from .data_sources import (fetch_news, ddg_news, resolve_company, count_mentions,
                           get_snapshot, score_headlines)
from .scoring import metric_specs, score
from .store import store
from .utils import as_list, clean_num

def _by_source(src):
    return [m for _, m, s in metric_specs() if s.get("source") == src]

def _targets(tickers):
    return as_list(tickers) or list(store().load("candidates", {}).keys())

# ---------------- 1. News ----------------
class NewsIn(BaseModel):
    query: str = Field(..., description="A news search query (topic, theme, event or sector).")

class SearchNewsTool(BaseTool):
    name: str = "search_live_news"
    description: str = ("Search live financial news. Returns headlines with id, title and publisher. "
                        "Call several times with different queries.")
    args_schema: Type[BaseModel] = NewsIn

    def _run(self, query: str) -> str:
        st = store()
        pool = st.load("headlines", [])
        known = {h["title"] for h in pool}
        fresh = []
        for h in fetch_news(query):
            if h["title"] not in known:
                h["id"] = len(pool) + len(fresh)
                fresh.append(h)
        pool.extend(fresh)
        st.save("headlines", pool)
        cap = SETTINGS["news"]["max_headlines_returned"]
        shown = (fresh or pool[-cap:])[:cap]
        return json.dumps([{"id": h["id"], "title": h["title"][:140], "publisher": h["publisher"]}
                           for h in shown])

# ---------------- 2. Resolver ----------------
class ResolveIn(BaseModel):
    names: Union[List[str], str] = Field(..., description="Company names found in headlines.")

class ResolveTickersTool(BaseTool):
    name: str = "resolve_tickers"
    description: str = "Resolve company names to verified exchange-listed tickers and measure news attention."
    args_schema: Type[BaseModel] = ResolveIn

    def _run(self, names) -> str:
        st = store()
        heads = st.load("headlines", [])
        found, unresolved = {}, []
        for n in as_list(names):
            try:
                r = resolve_company(n)
            except Exception:
                r = None
            if not r:
                unresolved.append(n)
                continue
            mentions, pubs = count_mentions([n, r["company"]], heads)
            r.update(mentions=mentions, publishers=pubs)
            if r["ticker"] not in found or mentions > found[r["ticker"]]["mentions"]:
                found[r["ticker"]] = r
        ranked = sorted(found.values(), key=lambda r: (r["publishers"], r["mentions"]), reverse=True)
        ranked = ranked[: SETTINGS["discovery"]["max_candidates"]]
        st.save("candidates", {r["ticker"]: r for r in ranked})
        return json.dumps({"resolved": [{k: r[k] for k in ("ticker", "company", "mentions", "publishers")}
                                        for r in ranked], "unresolved": unresolved})

# ---------------- 3. Fundamentals ----------------
class TickersIn(BaseModel):
    tickers: Optional[Union[List[str], str]] = Field(
        default=None, description="Tickers to analyze. Omit to use all resolved candidates.")

class FundamentalsTool(BaseTool):
    name: str = "analyze_fundamentals"
    description: str = "Fetch valuation, quality, growth and analyst metrics for candidates."
    args_schema: Type[BaseModel] = TickersIn

    def _run(self, tickers=None) -> str:
        st = store()
        tick = _targets(tickers)
        if not tick:
            return "ERROR: no candidates available. Resolve tickers first."
        out, summary, keys = st.load("fundamentals", {}), [], _by_source("info")
        for t in tick:
            try:
                snap = get_snapshot(t)
                info = snap["info"]
                m = {k: clean_num(info.get(k)) for k in keys}
                price = clean_num(info.get("currentPrice")) or snap.get("last_close")
                tgt = clean_num(info.get("targetMeanPrice"))
                m["upside_to_target"] = (tgt / price - 1) if (price and tgt) else None
                meta = {"name": info.get("longName") or info.get("shortName"),
                        "sector": info.get("sector"), "industry": info.get("industry"),
                        "currency": info.get("currency")}
                out[t] = {"metrics": m, "meta": meta}
                summary.append({"ticker": t, "sector": meta["sector"],
                                "found": sum(v is not None for v in m.values()), "of": len(m)})
            except Exception as e:
                summary.append({"ticker": t, "error": str(e)[:120]})
        st.save("fundamentals", out)
        return json.dumps(summary)

# ---------------- 4. Market & risk ----------------
class MarketRiskTool(BaseTool):
    name: str = "analyze_market_risk"
    description: str = "Compute momentum, volatility, drawdown and news sentiment for candidates."
    args_schema: Type[BaseModel] = TickersIn

    def _run(self, tickers=None) -> str:
        st = store()
        cands = st.load("candidates", {})
        tick = _targets(tickers)
        if not tick:
            return "ERROR: no candidates available."
        out, summary = st.load("market_risk", {}), []
        for t in tick:
            try:
                snap = get_snapshot(t)
                m = {k: v for k, v in snap["tech"].items()}
                name = (cands.get(t) or {}).get("company") or snap["info"].get("longName") or t
                news = fetch_news(f"{name} stock", SETTINGS["news"]["company_news_items"])
                m["news_sentiment"] = score_headlines([h["title"] for h in news])
                out[t] = {"metrics": m,
                          "news": [{"title": h["title"][:140], "publisher": h["publisher"]} for h in news[:6]],
                          "news_publishers": sorted({h["publisher"] for h in news if h["publisher"]})}
                summary.append({"ticker": t, "metrics": sum(v is not None for v in m.values()),
                                "headlines": len(news)})
            except Exception as e:
                summary.append({"ticker": t, "error": str(e)[:120]})
        st.save("market_risk", out)
        return json.dumps(summary)

# ---------------- 5. Validation ----------------
class ValidateTool(BaseTool):
    name: str = "validate_candidates"
    description: str = "Check price consistency, data completeness and multi-source news corroboration."
    args_schema: Type[BaseModel] = TickersIn

    def _run(self, tickers=None) -> str:
        st = store()
        tick = _targets(tickers)
        fund, mr, cands = st.load("fundamentals", {}), st.load("market_risk", {}), st.load("candidates", {})
        n_specs = len(metric_specs())
        raw = {}
        for t in tick:
            issues = []
            snap = get_snapshot(t)
            price, last = clean_num(snap["info"].get("currentPrice")), snap.get("last_close")
            price_ok = None
            if price and last:
                diff = abs(price - last) / last
                price_ok = diff <= SETTINGS["validation"]["price_tolerance"]
                if not price_ok:
                    issues.append(f"quote vs last close differ by {diff:.1%}")
            else:
                issues.append("price cross-check unavailable")
            metrics = {**fund.get(t, {}).get("metrics", {}), **mr.get(t, {}).get("metrics", {})}
            completeness = sum(v is not None for v in metrics.values()) / n_specs
            name = (cands.get(t) or {}).get("company") or t
            second = ddg_news(f"{name} stock")
            pubs = set(mr.get(t, {}).get("news_publishers", [])) | {h["publisher"] for h in second if h["publisher"]}
            second_confirms = bool(second)
            if not second_confirms:
                issues.append("no independent confirmation from second news source")
            raw[t] = {"price_ok": price_ok, "completeness": completeness, "n_publishers": len(pubs),
                      "second_source": second_confirms, "issues": issues}
        # corroboration is scored relative to peers (no absolute threshold)
        s = pd.Series({t: r["n_publishers"] for t, r in raw.items()}, dtype=float)
        corr = ((s.rank() - 1) / (len(s) - 1)) if len(s) > 1 else pd.Series(0.5, index=s.index)
        for t, r in raw.items():
            parts = [r["completeness"], float(corr[t]), float(r["second_source"])]
            if r["price_ok"] is not None:
                parts.append(float(r["price_ok"]))
            r["validation_score"] = round(sum(parts) / len(parts), 3)
        st.save("validation", raw)
        return json.dumps({t: {"score": r["validation_score"], "issues": r["issues"]} for t, r in raw.items()})

# ---------------- 6. Ranking ----------------
class RankIn(BaseModel):
    top_n: Optional[int] = Field(default=None, description="How many top results to return.")

class RankTool(BaseTool):
    name: str = "rank_candidates"
    description: str = "Rank all candidates with the deterministic percentile scoring engine."
    args_schema: Type[BaseModel] = RankIn

    def _run(self, top_n=None) -> str:
        st = store()
        profile = st.load("profile", {})
        cands = st.load("candidates", {})
        fund, mr, val = st.load("fundamentals", {}), st.load("market_risk", {}), st.load("validation", {})
        rows = {}
        for t in cands:
            if t not in fund and t not in mr:
                continue
            r = {**fund.get(t, {}).get("metrics", {}), **mr.get(t, {}).get("metrics", {}),
                 "news_mentions": cands[t]["mentions"], "news_publishers": cands[t]["publishers"]}
            rows[t] = r
        if not rows:
            return "ERROR: no analyzed candidates to rank."
        df = pd.DataFrame.from_dict(rows, orient="index")
        groups = pd.Series({t: fund.get(t, {}).get("meta", {}).get("sector") for t in rows})
        vscores = {t: v["validation_score"] for t, v in val.items()}
        ranking = score(df, groups, vscores, profile.get("family_weights"))
        for r in ranking:
            r["company"] = cands[r["ticker"]]["company"]
        st.save("ranking", ranking)
        n = top_n or profile.get("top_n", 3)
        return json.dumps([{k: r[k] for k in ("rank", "ticker", "company", "final", "composite",
                                              "coverage", "validation", "families")}
                           for r in ranking[:n]])

# ---------------- 7. Evidence ----------------
class EvidenceTool(BaseTool):
    name: str = "get_evidence"
    description: str = "Get metrics, strengths, weaknesses, risk flags, headlines and validation issues for top-ranked tickers."
    args_schema: Type[BaseModel] = TickersIn

    def _run(self, tickers=None) -> str:
        st = store()
        ranking = st.load("ranking", [])
        n = st.load("profile", {}).get("top_n", 3)
        wanted = as_list(tickers) or [r["ticker"] for r in ranking[:n]]
        mr, val = st.load("market_risk", {}), st.load("validation", {})
        out = []
        for r in ranking:
            if r["ticker"] in wanted:
                t = r["ticker"]
                out.append({"rank": r["rank"], "ticker": t, "company": r["company"],
                            "final": r["final"], "coverage": r["coverage"], "validation": r["validation"],
                            "strengths": r["strengths"], "weaknesses": r["weaknesses"],
                            "risk_flags": r["risk_flags"], "key_metrics": r["raw"],
                            "recent_headlines": mr.get(t, {}).get("news", [])[:4],
                            "validation_issues": val.get(t, {}).get("issues", [])})
        return json.dumps(out)

TOOLS = {t.name: t for t in [SearchNewsTool(), ResolveTickersTool(), FundamentalsTool(),
                              MarketRiskTool(), ValidateTool(), RankTool(), EvidenceTool()]}

Overwriting /content/stock_platform/src/stockcrew/tools.py


In [15]:
%%writefile /content/stock_platform/src/stockcrew/llm.py
import os
import requests
from crewai import LLM
from .config import SETTINGS

L = SETTINGS["llm"]

def _ollama_up():
    try:
        return requests.get(L["ollama_base_url"] + "/api/tags", timeout=2).ok
    except Exception:
        return False

def build_llms(provider):
    """Returns (worker_llm, manager_llm) for a provider, or None if unavailable."""
    common = dict(temperature=L["temperature"], max_tokens=L["max_tokens"])
    if provider == "groq" and os.getenv("GROQ_API_KEY"):
        k = os.environ["GROQ_API_KEY"]
        return (LLM(model=L["groq_worker_model"], api_key=k, **common),
                LLM(model=L["groq_manager_model"], api_key=k, **common))
    if provider == "ollama" and _ollama_up():
        mk = lambda: LLM(model=L["ollama_model"], base_url=L["ollama_base_url"], **common)
        return mk(), mk()
    if provider == "huggingface" and os.getenv("HF_TOKEN"):
        os.environ["HUGGINGFACE_API_KEY"] = os.environ["HF_TOKEN"]
        mk = lambda: LLM(model=L["hf_model"], api_key=os.environ["HF_TOKEN"], **common)
        return mk(), mk()
    return None

Overwriting /content/stock_platform/src/stockcrew/llm.py


In [16]:
%%writefile /content/stock_platform/src/stockcrew/crew_factory.py
from crewai import Agent, Task, Crew, Process
from .config import SETTINGS, load_yaml
from .schemas import FinalRecommendation
from .store import store
from .tools import TOOLS

def build_crew(llm_worker, llm_manager, verbose=True):
    A, T = load_yaml("agents.yaml"), load_yaml("tasks.yaml")
    agents = {}
    for key, spec in A.items():
        if key == "manager":
            continue
        agents[key] = Agent(
            role=spec["role"], goal=spec["goal"], backstory=spec["backstory"],
            tools=[TOOLS[n] for n in spec.get("tools", [])], llm=llm_worker,
            allow_delegation=False, max_iter=SETTINGS["crew"]["max_iter"], verbose=verbose)
    m = A["manager"]
    manager = Agent(role=m["role"], goal=m["goal"], backstory=m["backstory"],
                    llm=llm_manager, allow_delegation=True, verbose=verbose)
    tasks = {}
    for key, spec in T.items():
        kw = dict(description=spec["description"], expected_output=spec["expected_output"],
                  context=[tasks[c] for c in spec.get("context", [])])
        if key == "report":
            kw["output_pydantic"] = FinalRecommendation
        tasks[key] = Task(**kw)       # no fixed agent: the manager delegates
    return Crew(
        agents=list(agents.values()), tasks=list(tasks.values()),
        process=Process.hierarchical, manager_agent=manager,   # manager is NOT in `agents`
        max_rpm=SETTINGS["crew"]["max_rpm"], memory=False, planning=False,
        verbose=verbose, output_log_file=str(store().dir / "crew.log"))

Overwriting /content/stock_platform/src/stockcrew/crew_factory.py


In [17]:
%%writefile /content/stock_platform/src/stockcrew/run.py
import json, re, traceback
from .config import SETTINGS
from .crew_factory import build_crew
from .llm import build_llms
from .schemas import FinalRecommendation
from .store import init_run

def _extract(result):
    p = getattr(result, "pydantic", None)
    if p is not None:
        return p.model_dump()
    raw = getattr(result, "raw", "") or str(result)
    m = re.search(r"\{.*\}", raw, re.S)
    if m:
        try:
            return FinalRecommendation(**json.loads(m.group(0))).model_dump()
        except Exception:
            pass
    return None

def _deterministic(top):
    return {"recommended_ticker": top["ticker"], "company_name": top["company"],
            "confidence": round(top["coverage"] * top["validation"], 3),
            "thesis": "Top-ranked by the quantitative engine (LLM narrative unavailable).",
            "supporting_evidence": top["strengths"], "key_risks": top["risk_flags"] + top["weaknesses"],
            "runner_ups": []}

def _render(final, ranking, meta):
    L = [f"# Recommendation: {final['company_name']} ({final['recommended_ticker']})",
         f"*Run {meta['run_id']} | focus: {meta['focus']} | {meta['risk_profile']} | {meta['horizon']}*", "",
         f"**LLM confidence:** {final['confidence']:.2f}  |  **Data confidence (coverage x validation):** {final['data_confidence']:.2f}",
         f"**Agrees with quantitative ranker:** {final['agrees_with_ranker']}", "", "## Thesis", final["thesis"],
         "", "## Supporting evidence"] + [f"- {e}" for e in final["supporting_evidence"]]
    L += ["", "## Key risks"] + [f"- {r}" for r in final["key_risks"]]
    L += ["", "## Ranking", "| Rank | Ticker | Company | Final | Coverage | Validation |", "|---|---|---|---|---|---|"]
    L += [f"| {r['rank']} | {r['ticker']} | {r['company']} | {r['final']} | {r['coverage']} | {r['validation']} |"
          for r in ranking]
    L += ["", "> Educational research output, not investment advice. Data from free, unofficial sources may be delayed or incomplete."]
    return "\n".join(L)

def run(focus="", region=None, horizon="medium-term", risk_profile="balanced",
        top_n=3, family_weights=None, verbose=True):
    """family_weights example: {"risk": 2, "growth": 0.5} to express investor preferences."""
    st = init_run()
    st.save("profile", {"family_weights": family_weights or {}, "top_n": top_n})
    meta = {"run_id": st.run_id, "focus": focus or "any sector or theme currently in the news",
            "region": region or SETTINGS["news"]["region"], "horizon": horizon, "risk_profile": risk_profile}
    result, errors = None, []
    for provider in SETTINGS["llm"]["provider_order"]:
        llms = build_llms(provider)
        if not llms:
            continue
        try:
            crew = build_crew(*llms, verbose=verbose)
            result = crew.kickoff(inputs=meta)
            if st.load("ranking"):
                break
        except Exception as e:
            errors.append(f"{provider}: {e}")
            traceback.print_exc()
    ranking = st.load("ranking", [])
    if not ranking:
        raise RuntimeError(f"Pipeline produced no ranking. Errors: {errors}")
    top = ranking[0]
    final = (_extract(result) if result else None) or _deterministic(top)
    final["ranker_top"] = top["ticker"]
    final["agrees_with_ranker"] = final["recommended_ticker"].upper() == top["ticker"].upper()
    final["data_confidence"] = round(top["coverage"] * top["validation"], 3)
    report = _render(final, ranking, meta)
    (st.dir / "report.md").write_text(report)
    st.save("final", final)
    return {"final": final, "ranking": ranking, "report_md": report, "run_dir": str(st.dir)}

Overwriting /content/stock_platform/src/stockcrew/run.py


In [18]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.


In [19]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"]); time.sleep(6)
!ollama pull llama3.1:8b

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [20]:
!pip install -U litellm


  Using cached litellm-1.104.0-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (43 kB)
Using cached litellm-1.104.0-cp310-abi3-manylinux_2_28_x86_64.whl (36.8 MB)
  Attempting uninstall: litellm
    Found existing installation: litellm 1.74.9
    Uninstalling litellm-1.74.9:
      Successfully uninstalled litellm-1.74.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 0.203.2 requires litellm==1.74.9, but you have litellm 1.104.0 which is incompatible.


In [21]:
!pip install nest_asyncio

In [ ]:
import importlib, stockcrew.config
from stockcrew.run import run
from IPython.display import Markdown, display

out = run(
    focus="",                      # "" = any sector, or e.g. "semiconductors", "renewable energy", "Indian banks"
    risk_profile="balanced",       # free text, flows into prompts
    horizon="medium-term",
    top_n=3,
    family_weights={"risk": 1.5},  # optional investor preference; omit for equal weights
)
display(Markdown(out["report_md"]))

In [22]:
import crewai
print(crewai.__version__)

1.15.23


In [23]:
!pip show crewai crewai-tools litellm

Name: crewai
Version: 1.15.23
Summary: Cutting-edge framework for orchestrating role-playing, autonomous AI agents. By fostering collaborative intelligence, CrewAI empowers agents to work together seamlessly, tackling complex tasks.
Home-page: https://crewai.com
Author: 
Author-email: Joao Moura <joao@crewai.com>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiofiles, aiosqlite, appdirs, cel-python, chromadb, click, crewai-cli, crewai-core, httpx, instructor, json-repair, json5, jsonref, lancedb, mcp, openai, openpyxl, opentelemetry-api, opentelemetry-exporter-otlp-proto-http, opentelemetry-sdk, pdfplumber, portalocker, pydantic, pydantic-settings, pyjwt, python-dotenv, pyyaml, regex, tokenizers, tomli, tomli-w
Required-by: 
---
Name: litellm
Version: 1.104.0
Summary: Library to easily interface with LLM API providers
Home-page: https://litellm.ai
Author: BerriAI
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiohttp, boto3,

In [24]:
!pip uninstall -y crewai crewai-tools

Found existing installation: crewai 1.15.23
Uninstalling crewai-1.15.23:
  Successfully uninstalled crewai-1.15.23


In [25]:
!pip install "crewai==0.95.0"
!pip install "crewai-tools==0.33.0"

ERROR: Ignored the following yanked versions: 0.165.0, 1.10.0, 1.12.0, 1.14.0
ERROR: Ignored the following versions that require a different python version: 0.100.0 Requires-Python <3.13,>=3.10; 0.100.1 Requires-Python <3.13,>=3.10; 0.102.0 Requires-Python <3.13,>=3.10; 0.105.0 Requires-Python <3.13,>=3.10; 0.108.0 Requires-Python <3.13,>=3.10; 0.114.0 Requires-Python <3.13,>=3.10; 0.117.0 Requires-Python <3.13,>=3.10; 0.117.1 Requires-Python <3.13,>=3.10; 0.118.0 Requires-Python <3.13,>=3.10; 0.119.0 Requires-Python <3.13,>=3.10; 0.120.0 Requires-Python <3.13,>=3.10; 0.120.1 Requires-Python <3.13,>=3.10; 0.121.0 Requires-Python <3.13,>=3.10; 0.121.1 Requires-Python <3.13,>=3.10; 0.14.0 Requires-Python >=3.10,<=3.13; 0.14.0rc0 Requires-Python >=3.10,<3.12; 0.14.0rc1 Requires-Python >=3.10,<=3.13; 0.14.1 Requires-Python >=3.10,<=3.13; 0.14.3 Requires-Python >=3.10,<=3.13; 0.14.4 Requires-Python >=3.10,<=3.13; 0.16.0 Requires-Python >=3.10,<=3.13; 0.16.1 Requires-Python >=3.10,<=3.13; 0.

In [26]:
import crewai
print(crewai.__version__)

1.15.23


In [27]:
!pip uninstall -y crewai crewai-tools

In [28]:
!pip install "crewai==0.102.0"
!pip install "crewai-tools==0.36.0"

ERROR: Ignored the following yanked versions: 0.165.0, 1.10.0, 1.12.0, 1.14.0
ERROR: Ignored the following versions that require a different python version: 0.100.0 Requires-Python <3.13,>=3.10; 0.100.1 Requires-Python <3.13,>=3.10; 0.102.0 Requires-Python <3.13,>=3.10; 0.105.0 Requires-Python <3.13,>=3.10; 0.108.0 Requires-Python <3.13,>=3.10; 0.114.0 Requires-Python <3.13,>=3.10; 0.117.0 Requires-Python <3.13,>=3.10; 0.117.1 Requires-Python <3.13,>=3.10; 0.118.0 Requires-Python <3.13,>=3.10; 0.119.0 Requires-Python <3.13,>=3.10; 0.120.0 Requires-Python <3.13,>=3.10; 0.120.1 Requires-Python <3.13,>=3.10; 0.121.0 Requires-Python <3.13,>=3.10; 0.121.1 Requires-Python <3.13,>=3.10; 0.14.0 Requires-Python >=3.10,<=3.13; 0.14.0rc0 Requires-Python >=3.10,<3.12; 0.14.0rc1 Requires-Python >=3.10,<=3.13; 0.14.1 Requires-Python >=3.10,<=3.13; 0.14.3 Requires-Python >=3.10,<=3.13; 0.14.4 Requires-Python >=3.10,<=3.13; 0.16.0 Requires-Python >=3.10,<=3.13; 0.16.1 Requires-Python >=3.10,<=3.13; 0.

In [30]:
import importlib, stockcrew.config
from stockcrew.run import run
from IPython.display import Markdown, display

out = run(
    focus="",                      # "" = any sector, or e.g. "semiconductors", "renewable energy", "Indian banks"
    risk_profile="balanced",       # free text, flows into prompts
    horizon="medium-term",
    top_n=3,
    family_weights={"risk": 1.5},  # optional investor preference; omit for equal weights
)
display(Markdown(out["report_md"]))

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/crewai/utilities/i18n.py", line 43, in load_prompts
FileNotFoundError: [Errno 2] No such file or directory: '/usr/local/lib/python3.13/dist-packages/crewai/utilities/../translations/en.json'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/stock_platform/src/stockcrew/run.py", line 54, in run
    crew = build_crew(*llms, verbose=verbose)
  File "/content/stock_platform/src/stockcrew/crew_factory.py", line 13, in build_crew
    agents[key] = Agent(
                  ~~~~~^
        role=spec["role"], goal=spec["goal"], backstory=spec["backstory"],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        tools=[TOOLS[n] for n in spec.get("tools", [])], llm=llm_worker,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        allow_delegation=False, max_iter=SETTINGS["crew"]["max_iter"],

RuntimeError: Pipeline produced no ranking. Errors: ["groq: Prompt file 'None' not found.", "ollama: Prompt file 'None' not found."]

In [22]:
pip uninstall -y crewai crewai-core crewai-cli

Found existing installation: crewai 0.203.2
Uninstalling crewai-0.203.2:
  Successfully uninstalled crewai-0.203.2


In [23]:
pip show crewai

In [24]:
!pip install "crewai==0.203.2"

  Using cached crewai-0.203.2-py3-none-any.whl.metadata (35 kB)
  Using cached litellm-1.74.9-py3-none-any.whl.metadata (40 kB)
Using cached crewai-0.203.2-py3-none-any.whl (473 kB)
Using cached litellm-1.74.9-py3-none-any.whl (8.7 MB)
  Attempting uninstall: litellm
    Found existing installation: litellm 1.104.0
    Uninstalling litellm-1.104.0:
      Successfully uninstalled litellm-1.104.0


In [32]:
!pip index versions crewai

crewai (1.15.23)
Available versions: 1.15.23, 1.15.22, 1.15.21, 1.15.20, 1.15.19, 1.15.18, 1.15.17, 1.15.16, 1.15.15, 1.15.14, 1.15.13, 1.15.12, 1.15.11, 1.15.10, 1.15.9, 1.15.8, 1.15.7, 1.15.6, 1.15.5, 1.15.4, 1.15.3, 1.15.2, 1.15.1, 1.15.0, 1.14.7, 1.14.6, 1.14.5, 1.14.4, 1.14.3, 1.14.2, 1.14.1, 1.13.0, 1.12.2, 1.12.1, 1.11.1, 1.11.0, 1.10.1, 1.9.3, 1.9.2, 1.9.1, 1.9.0, 1.8.1, 1.8.0, 1.7.2, 1.7.1, 1.7.0, 1.6.1, 1.6.0, 1.5.0, 1.4.1, 1.4.0, 1.3.0, 1.2.1, 1.2.0, 1.1.0, 1.0.0, 0.203.2, 0.203.1, 0.203.0, 0.201.1, 0.201.0, 0.193.2, 0.193.1, 0.193.0, 0.186.1, 0.186.0, 0.177.0, 0.175.0, 0.165.1, 0.159.0, 0.157.0, 0.152.0, 0.150.0, 0.148.0, 0.141.0, 0.140.0, 0.134.0, 0.130.0, 0.126.0, 0.11.2, 0.11.1, 0.11.0, 0.10.0, 0.5.5, 0.5.3, 0.5.2, 0.5.0, 0.1.32, 0.1.24, 0.1.23, 0.1.17, 0.1.16, 0.1.15, 0.1.14, 0.1.7, 0.1.6, 0.1.5, 0.1.3, 0.1.2, 0.1.1, 0.1.0


In [32]:
!pip uninstall -y crewai crewai-core crewai-cli

Found existing installation: crewai 0.203.2
Uninstalling crewai-0.203.2:
  Successfully uninstalled crewai-0.203.2


In [33]:
!pip show crewai

In [34]:
!pip install "crewai==0.203.2"

  Using cached crewai-0.203.2-py3-none-any.whl.metadata (35 kB)
Using cached crewai-0.203.2-py3-none-any.whl (473 kB)


In [25]:
import crewai
print(crewai.__version__)

0.203.2


In [36]:
import importlib, stockcrew.config
from stockcrew.run import run
from IPython.display import Markdown, display

out = run(
    focus="",                      # "" = any sector, or e.g. "semiconductors", "renewable energy", "Indian banks"
    risk_profile="balanced",       # free text, flows into prompts
    horizon="medium-term",
    top_n=3,
    family_weights={"risk": 1.5},  # optional investor preference; omit for equal weights
)
display(Markdown(out["report_md"]))

KeyboardInterrupt: 

In [37]:
!pip uninstall -y crewai litellm

Found existing installation: crewai 0.203.2
Uninstalling crewai-0.203.2:
  Successfully uninstalled crewai-0.203.2
Found existing installation: litellm 1.74.9
Uninstalling litellm-1.74.9:
  Successfully uninstalled litellm-1.74.9


In [40]:
!pip install "crewai==0.203.2"
!pip install "litellm==1.51.3"

ERROR: Ignored the following yanked versions: 0.1.2291, 0.1.7701, 0.1.7713
ERROR: Could not find a version that satisfies the requirement litellm==1.51.3 (from versions: 0.1.0, 0.1.1, 0.1.2, 0.1.3, 0.1.31, 0.1.32, 0.1.34, 0.1.201, 0.1.202, 0.1.203, 0.1.204, 0.1.205, 0.1.206, 0.1.207, 0.1.208, 0.1.209, 0.1.210, 0.1.211, 0.1.212, 0.1.213, 0.1.214, 0.1.215, 0.1.216, 0.1.217, 0.1.218, 0.1.219, 0.1.220, 0.1.221, 0.1.222, 0.1.223, 0.1.224, 0.1.225, 0.1.226, 0.1.227, 0.1.228, 0.1.229, 0.1.230, 0.1.231, 0.1.232, 0.1.233, 0.1.234, 0.1.235, 0.1.236, 0.1.237, 0.1.238, 0.1.330, 0.1.331, 0.1.341, 0.1.343, 0.1.345, 0.1.347, 0.1.348, 0.1.349, 0.1.351, 0.1.352, 0.1.353, 0.1.354, 0.1.356, 0.1.360, 0.1.361, 0.1.362, 0.1.363, 0.1.364, 0.1.365, 0.1.366, 0.1.367, 0.1.368, 0.1.369, 0.1.370, 0.1.371, 0.1.372, 0.1.373, 0.1.375, 0.1.376, 0.1.379, 0.1.380, 0.1.381, 0.1.383, 0.1.384, 0.1.385, 0.1.386, 0.1.387, 0.1.388, 0.1.389, 0.1.392, 0.1.393, 0.1.394, 0.1.398, 0.1.399, 0.1.400, 0.1.401, 0.1.402, 0.1.403, 0.1.

In [43]:
!pip install "crewai==0.203.2"
!pip install "litellm==1.51.3"

ERROR: Ignored the following yanked versions: 0.1.2291, 0.1.7701, 0.1.7713
ERROR: Could not find a version that satisfies the requirement litellm==1.51.3 (from versions: 0.1.0, 0.1.1, 0.1.2, 0.1.3, 0.1.31, 0.1.32, 0.1.34, 0.1.201, 0.1.202, 0.1.203, 0.1.204, 0.1.205, 0.1.206, 0.1.207, 0.1.208, 0.1.209, 0.1.210, 0.1.211, 0.1.212, 0.1.213, 0.1.214, 0.1.215, 0.1.216, 0.1.217, 0.1.218, 0.1.219, 0.1.220, 0.1.221, 0.1.222, 0.1.223, 0.1.224, 0.1.225, 0.1.226, 0.1.227, 0.1.228, 0.1.229, 0.1.230, 0.1.231, 0.1.232, 0.1.233, 0.1.234, 0.1.235, 0.1.236, 0.1.237, 0.1.238, 0.1.330, 0.1.331, 0.1.341, 0.1.343, 0.1.345, 0.1.347, 0.1.348, 0.1.349, 0.1.351, 0.1.352, 0.1.353, 0.1.354, 0.1.356, 0.1.360, 0.1.361, 0.1.362, 0.1.363, 0.1.364, 0.1.365, 0.1.366, 0.1.367, 0.1.368, 0.1.369, 0.1.370, 0.1.371, 0.1.372, 0.1.373, 0.1.375, 0.1.376, 0.1.379, 0.1.380, 0.1.381, 0.1.383, 0.1.384, 0.1.385, 0.1.386, 0.1.387, 0.1.388, 0.1.389, 0.1.392, 0.1.393, 0.1.394, 0.1.398, 0.1.399, 0.1.400, 0.1.401, 0.1.402, 0.1.403, 0.1.

In [44]:
import crewai
import litellm

print(crewai.__version__)
print(litellm.__version__)

AttributeError: partially initialized module 'litellm' from '/usr/local/lib/python3.13/dist-packages/litellm/__init__.py' has no attribute 'litellm_core_utils' (most likely due to a circular import)

In [26]:
import crewai
print("CrewAI =", crewai.__version__)

import pkg_resources
print("LiteLLM =", pkg_resources.get_distribution("litellm").version)

CrewAI = 0.203.2


/tmp/ipykernel_10807/4039699252.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


LiteLLM = 1.74.9


In [46]:
!pip uninstall -y crewai crewai-core crewai-cli litellm

Found existing installation: crewai 0.203.2
Uninstalling crewai-0.203.2:
  Successfully uninstalled crewai-0.203.2
Found existing installation: litellm 1.74.9
Uninstalling litellm-1.74.9:
  Successfully uninstalled litellm-1.74.9


In [47]:
!pip show crewai
!pip show litellm

In [52]:
!pip install crewai==0.203.2

In [27]:
import crewai
import pkg_resources

print("CrewAI =", crewai.__version__)
print("LiteLLM =", pkg_resources.get_distribution("litellm").version)


CrewAI = 0.203.2
LiteLLM = 1.74.9


In [9]:
import importlib, stockcrew.config
from stockcrew.run import run
from IPython.display import Markdown, display

out = run(
    focus="",                      # "" = any sector, or e.g. "semiconductors", "renewable energy", "Indian banks"
    risk_profile="balanced",       # free text, flows into prompts
    horizon="medium-term",
    top_n=3,
    family_weights={"risk": 1.5},  # optional investor preference; omit for equal weights
)
display(Markdown(out["report_md"]))

ModuleNotFoundError: No module named 'stockcrew'

In [53]:
import crewai
import pkg_resources

print(crewai.__version__)
print(pkg_resources.get_distribution("litellm").version)

1.15.23
1.74.9


In [2]:
from pathlib import Path

f = Path("/content/stock_platform/config/settings.yaml")

txt = f.read_text()

txt = txt.replace(
    "groq/llama-3.3-70b-versatile",
    "groq/openai/gpt-oss-120b"
)

f.write_text(txt)

print("Updated settings.yaml")

Updated settings.yaml


In [3]:
from pathlib import Path

print(Path("/content/stock_platform/config/settings.yaml").read_text())

llm:
  provider_order: [groq, ollama, huggingface]   # tried in order, auto-fallback on failure
  groq_worker_model: groq/openai/gpt-oss-120b   # switch to a smaller model if you hit rate limits
  groq_manager_model: groq/openai/gpt-oss-120b
  ollama_model: ollama/llama3.1:8b
  ollama_base_url: http://localhost:11434
  hf_model: huggingface/Qwen/Qwen2.5-72B-Instruct   # availability on free HF inference changes; verify
  temperature: 0.1
  max_tokens: 2048
crew:
  max_rpm: 15
  max_iter: 8
news:
  region: US
  language: en
  lookback: 3d
  max_items_per_query: 25
  max_headlines_returned: 40
  company_news_items: 15
  cache_ttl_minutes: 30
discovery:
  max_candidates: 8
  name_match_min: 0.6
market:
  history_period: 1y
  interval: 1d
  trading_days: 252
  trend_sma_days: 200
  windows: {ret_1m: 21, ret_3m: 63, ret_6m: 126}
  cache_ttl_minutes: 60
validation:
  price_tolerance: 0.05
sentiment:
  backend: finbert          # finbert | none
  model: ProsusAI/finbert
scoring:
  min_peer_gr

In [6]:
from pathlib import Path

print(Path("/content/stock_platform/src/stockcrew/llm.py").read_text())

import os
import requests
from crewai import LLM
from .config import SETTINGS

L = SETTINGS["llm"]

def _ollama_up():
    try:
        return requests.get(L["ollama_base_url"] + "/api/tags", timeout=2).ok
    except Exception:
        return False

def build_llms(provider):
    """Returns (worker_llm, manager_llm) for a provider, or None if unavailable."""
    common = dict(temperature=L["temperature"], max_tokens=L["max_tokens"])
    if provider == "groq" and os.getenv("GROQ_API_KEY"):
        k = os.environ["GROQ_API_KEY"]
        return (LLM(model=L["groq_worker_model"], api_key=k, **common),
                LLM(model=L["groq_manager_model"], api_key=k, **common))
    if provider == "ollama" and _ollama_up():
        mk = lambda: LLM(model=L["ollama_model"], base_url=L["ollama_base_url"], **common)
        return mk(), mk()
    if provider == "huggingface" and os.getenv("HF_TOKEN"):
        os.environ["HUGGINGFACE_API_KEY"] = os.environ["HF_TOKEN"]
        mk = lambda: LLM(model=L["

In [10]:
!ls /content

sample_data  stock_platform


In [11]:
!find /content/stock_platform -type d | grep stockcrew

/content/stock_platform/src/stockcrew
/content/stock_platform/src/stockcrew/__pycache__


In [12]:
import sys

sys.path.append("/content/stock_platform/src")

print("added")

added


In [13]:
import stockcrew

print("success")

success


In [14]:
import sys

project_path = "/content/stock_platform/src"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

import stockcrew

print("Imported stockcrew successfully")

Imported stockcrew successfully


In [20]:
!which ollama

/usr/local/bin/ollama


In [32]:
!nvidia-smi
!free -h


/bin/bash: line 1: nvidia-smi: command not found
               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.3Gi       6.7Gi        40Mi       3.1Gi       9.4Gi
Swap:             0B          0B          0B


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
###################################################                       71.2%

In [28]:
!ollama --version

ollama version is 0.40.0


In [29]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.1:8b    46e0c10c039e    4.9 GB    2 minutes ago    


In [30]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("✅ Ollama server started")


✅ Ollama server started


In [31]:
import requests

r = requests.get("http://localhost:11434/api/tags")

print("Status:", r.status_code)
print(r.json())

Status: 200
{'models': [{'name': 'llama3.1:8b', 'model': 'llama3.1:8b', 'modified_at': '2026-10-07T06:36:10.257270162Z', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools']}]}


In [32]:
from pathlib import Path
import re

f = Path("/content/stock_platform/config/settings.yaml")

txt = f.read_text(encoding="utf-8")

txt = re.sub(
    r"provider_order:.*",
    "provider_order: [ollama, groq, huggingface]",
    txt
)

txt = re.sub(
    r"ollama_model:.*",
    "ollama_model: ollama/llama3.1:8b",
    txt
)

txt = re.sub(
    r"max_tokens:\s*\d+",
    "max_tokens: 256",
    txt
)

txt = re.sub(
    r"max_iter:\s*\d+",
    "max_iter: 4",
    txt
)

f.write_text(txt, encoding="utf-8")

print("✅ settings.yaml updated")

✅ settings.yaml updated


In [33]:
import importlib
import stockcrew.config

importlib.reload(stockcrew.config)

print(stockcrew.config.SETTINGS["llm"])

{'provider_order': ['ollama', 'groq', 'huggingface'], 'groq_worker_model': 'groq/llama-3.3-70b-versatile', 'groq_manager_model': 'groq/llama-3.3-70b-versatile', 'ollama_model': 'ollama/llama3.1:8b', 'ollama_base_url': 'http://localhost:11434', 'hf_model': 'huggingface/Qwen/Qwen2.5-72B-Instruct', 'temperature': 0.1, 'max_tokens': 256}


In [34]:
from stockcrew.llm import build_llms

worker, manager = build_llms("ollama")

print(worker)
print(manager)

In [35]:
import importlib, stockcrew.config
from stockcrew.run import run
from IPython.display import Markdown, display

out = run(
    focus="",                      # "" = any sector, or e.g. "semiconductors", "renewable energy", "Indian banks"
    risk_profile="balanced",       # free text, flows into prompts
    horizon="medium-term",
    top_n=3,
    family_weights={"risk": 1.5},  # optional investor preference; omit for equal weights
)
display(Markdown(out["report_md"]))

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4e452069-8d10-482d-b950-e29dfe54c54c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer                                                                                │
│                                                                                                                 │
│  Task: Delegate to the Live Market News Scout. Use search_live_news with at least three DIFFERENT queries       │
│  derived from the focus "any sector or theme currently in the news" and region "US" (for example earnings,      │
│  guidance, regulation, product news, analyst actions, market movers; if the focus is broad, query generally).   │
│  Then list the distinct publicly traded companies that literally appear in the returned headlines.              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Live Market News Scout                                                                                  │
│                                                                                                                 │
│  Task: search_live_news                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: a767145a-4839-488d-abc0-35995e6eb5ed                                                                     │
│  Agent: Chief Investment Officer                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4e452069-8d10-482d-b950-e29dfe54c54c                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "recommended_ticker": "AMZN",                                                                                │
│    "company_name": "Amazon.com, Inc.",                                                                          │
│    "confidence": 0.6923,                                                                                        │
│    "thesis": "AMZN is a strong investment opportunity due to its high rank, strong coverage, and key            │
│  strengths.",                                                                                                   │
│    "supporting_evidence": [                                                                                     │
│      "High rank of 1",                                                                                          │
│      "Final score of 0.4239",                                                                                   │
│      "Strong coverage of 0.682",                                                                                │
│      "High recommendation mean",                                                                                │
│      "High earnings growth",                                                                                    │
│      "High upside to target"                                                                                    │
│    ],                                                                                                           │
│    "key_risks": [                                                                                               │
│      "Low profit margins",                                                                                      │
│      "Low operating margins",                                                                                   │
│      "Low return on equity"                                                                                     │
│    ],                                                                                                           │
│    "runner_ups": []                                                                                             │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

# Recommendation: Amazon.com, Inc. (AMZN)
*Run 20261007-064050 | focus: any sector or theme currently in the news | balanced | medium-term*

**LLM confidence:** 0.69  |  **Data confidence (coverage x validation):** 0.61
**Agrees with quantitative ranker:** True

## Thesis
AMZN is a strong investment opportunity due to its high rank, strong coverage, and key strengths.

## Supporting evidence
- High rank of 1
- Final score of 0.4239
- Strong coverage of 0.682
- High recommendation mean
- High earnings growth
- High upside to target

## Key risks
- Low profit margins
- Low operating margins
- Low return on equity

## Ranking
| Rank | Ticker | Company | Final | Coverage | Validation |
|---|---|---|---|---|---|
| 1 | AMZN | Amazon.com, Inc. | 0.4239 | 0.682 | 0.898 |
| 2 | TSLA | Tesla, Inc. | 0.236 | 0.682 | 1.0 |
| 3 | AAPL | Apple Inc. | 0.2039 | 0.682 | 0.648 |

> Educational research output, not investment advice. Data from free, unofficial sources may be delayed or incomplete.